# claim-rebuild (chapter 7)

Rebuild the vendor's 20 percent cellar loss claim on Kestrel's synthetic CSV, net of the licence and the cloud bill, then plot the waterfall (Figure 7.1).

In [ ]:
import pandas as pd
inp = pd.read_csv("../data/kestrel_inputs.csv").set_index("item")["value"]
inp

## The loss rate, from twelve months not three (7.1)

In [ ]:
loss = pd.read_csv("../data/kestrel_monthly_loss.csv")
print(loss.tail(3).to_string(index=False))
q = loss.tail(3)
print("Quarter:", q["lost_hL"].sum(), "/", q["cast_hL"].sum(), "=", round(100 * q["lost_hL"].sum() / q["cast_hL"].sum(), 1), "percent")
print("Twelve months:", loss["lost_hL"].sum(), "/", loss["cast_hL"].sum(), "=", round(100 * loss["lost_hL"].sum() / loss["cast_hL"].sum(), 2), "percent")

## One formula per saving (7.2)

Saving a year = volume x loss rate x share recovered x value per lost hL. The share recovered is the vendor's promise; everything else is yours.

In [ ]:
lost = inp["volume_hL_a_year"] * inp["cellar_loss_rate"]
recovered = lost * inp["share_recovered_claim"]
headline = recovered * inp["selling_price_usd_a_hL"]
replacement = recovered * inp["value_lost_hL_replacement_usd"]
upper = recovered * inp["value_lost_hL_upper_bound_usd"]
licence = 12 * inp["licence_usd_a_month"]
cloud = 12 * inp["cloud_bill_usd_a_month"]
ceiling = inp["excess_cellar_loss_rate"] / inp["cellar_loss_rate"]
print(f"Lost {lost:,.0f} hL; recovered {recovered:,.0f} hL")
print(f"Headline {headline:,.0f}; replacement {replacement:,.0f}; upper bound {upper:,.0f}")
print(f"Net at replacement {replacement - licence - cloud:,.0f}; at the upper bound {upper - licence - cloud:,.0f}")
print(f"Largest share any tool can recover {ceiling:.3f}; claim inside the ceiling: {inp['share_recovered_claim'] <= ceiling}")

## The worked example in three currencies (7.4)

Book rates: USD 1 = INR 90 and EUR 1 = USD 1.10. Rupees in Indian grouping, euros to the nearest hundred.

In [ ]:
def indian(n):
    s, sign = f"{abs(round(n)):d}", "-" if n < 0 else ""
    head, tail = s[:-3], s[-3:]
    while len(head) > 2:
        tail, head = head[-2:] + "," + tail, head[:-2]
    return sign + (head + "," + tail if head else tail)

lines = [("Vendor headline, at selling price", headline), ("Rebuilt at replacement cost", replacement),
         ("Licence", -licence), ("Cloud bill", -cloud), ("Net at replacement cost", replacement - licence - cloud),
         ("Net at the peak season upper bound", upper - licence - cloud)]
pd.DataFrame([(n, f"{v:,.0f}", indian(v * inp["usd_to_inr_book"]), f"{round(v / inp['eur_to_usd_book'], -2):,.0f}")
              for n, v in lines], columns=["Line", "USD", "INR", "EUR"])

## Figure 7.1: the waterfall

In [ ]:
import matplotlib.pyplot as plt

steps = [("Headline", headline, 0), ("Value at USD 30", replacement - headline, headline),
         ("Licence", -licence, replacement), ("Cloud bill", -cloud, replacement - licence)]
fig, ax = plt.subplots(figsize=(7, 3.5))
for i, (name, h, base) in enumerate(steps):
    ax.bar(i, h, bottom=base, color="#0b4f45" if i == 0 else "#b23a2e")
net = replacement - licence - cloud
ax.bar(len(steps), net, color="#b23a2e")
ax.bar(len(steps) + 1, upper - licence - cloud, color="#9e9e9e")
ax.set_xticks(range(len(steps) + 2), [s[0] for s in steps] + ["Net", "Net, upper bound"], rotation=20)
ax.axhline(0, color="grey", lw=0.8)
ax.set_ylabel("USD a year (synthetic)")
plt.tight_layout()
plt.show()